In [ ]:
# ============================================================
# 13. SAVE MODEL AND ARTIFACTS
# ============================================================

ARTIFACT_DIR = "global_unemployment_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

model_path = os.path.join(
    ARTIFACT_DIR,
    "best_unemployment_probability_model.pkl"
)

backup_model_path = os.path.join(
    ARTIFACT_DIR,
    "plan_b_backup_model.pkl"
)

joblib.dump(best_pipeline, model_path)
joblib.dump(backup_pipeline, backup_model_path)

metadata = {
    "best_model_name": best_model_name,
    "backup_model_name": backup_model_name,
    "random_state": RANDOM_STATE,
    "features": features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "model_results": results_df
}

metadata_path = os.path.join(
    ARTIFACT_DIR,
    "model_metadata.pkl"
)

joblib.dump(metadata, metadata_path)

print("Best model saved to:", model_path)
print("Plan B backup model saved to:", backup_model_path)
print("Metadata saved to:", metadata_path)

Best model saved to: global_unemployment_artifacts/best_unemployment_probability_model.pkl
Plan B backup model saved to: global_unemployment_artifacts/plan_b_backup_model.pkl
Metadata saved to: global_unemployment_artifacts/model_metadata.pkl


In [ ]:
# ============================================================
# SAFE PREDICTION WITH PLAN B
# ============================================================

def safe_predict_proba(input_df):
    try:
        probabilities = best_pipeline.predict_proba(input_df)
        model_used = best_model_name

    except Exception as error:
        print("Best model failed. Switching to Plan B.")
        print("Error:", error)

        if backup_pipeline is None:
            raise RuntimeError(
                "Best model failed and no Plan B model is available."
            )

        probabilities = backup_pipeline.predict_proba(input_df)
        model_used = backup_model_name

    return probabilities, model_used

In [ ]:

# ============================================================
# 14. PREDICTION FUNCTION
# ============================================================

def classify_risk(unemployment_probability):
    if unemployment_probability < 35:
        return "Low Unemployment Risk"
    elif unemployment_probability < 65:
        return "Moderate Employment Uncertainty"
    else:
        return "High Unemployment Risk"


def predict_unemployment_probability(input_dict):
    input_df = pd.DataFrame([input_dict])

    # Ensure all required columns exist and fill missing values correctly
    for col in features:
        if col not in input_df.columns:
            if col in categorical_features:
                input_df[col] = "Unknown"
            else:
                input_df[col] = data[col].median()

    input_df = input_df[features]

    # Plan B error handling is used here.
    # If the best model/pipeline fails, safe_predict_proba automatically uses the backup model.
    probabilities, model_used = safe_predict_proba(input_df)

    prob_unemployed = probabilities[0][1] * 100
    prob_employed = 100 - prob_unemployed

    prediction = classify_risk(prob_unemployed)

    reasons = []

    if input_dict.get("GPA", data["GPA"].median()) < 2.5:
        reasons.append("Low GPA")

    if input_dict.get("Internship_Experience", "Unknown") == "No":
        reasons.append("No internship experience")

    if input_dict.get("Years_Since_Graduation", 0) > 3:
        reasons.append("Long period since graduation")

    if input_dict.get("Education_Level", "Unknown") in ["High School", "Diploma"]:
        reasons.append("Lower education level")

    if len(reasons) > 0:
        explanation = (
            "The unemployment probability may be influenced by: "
            + ", ".join(reasons)
        )
    else:
        explanation = (
            "The profile contains factors associated with stronger employability."
        )

    return {
        "Employment Probability": f"{prob_employed:.2f}%",
        "Unemployment Probability": f"{prob_unemployed:.2f}%",
        "Risk Level": prediction,
        "Model Used": model_used,
        "Explanation": explanation
    }

# Example prediction using first test sample
sample_input = X_test.iloc[0].to_dict()
predict_unemployment_probability(sample_input)


{'Employment Probability': '99.99%',
 'Unemployment Probability': '0.01%',
 'Risk Level': 'Low Unemployment Risk',
 'Model Used': 'XGBoost',
 'Explanation': 'The profile contains factors associated with stronger employability.'}

In [ ]:

# ============================================================
# 15. LOAD AND STRESS TEST RESULTS
# ============================================================

def run_load_stress_test(number_of_requests=100):
    start_time = time.perf_counter()

    for i in range(number_of_requests):
        sample = X_test.sample(1, random_state=RANDOM_STATE + i).iloc[0].to_dict()
        _ = predict_unemployment_probability(sample)

    total_time = time.perf_counter() - start_time
    average_time = total_time / number_of_requests
    requests_per_second = number_of_requests / total_time

    print("Load and Stress Test Results")
    print(" Total time:", round(total_time, 4), "seconds")
    print(" Average time:", round(average_time, 6), "seconds/request")
    print(" Requests per second:", round(requests_per_second, 2))

    return {
        "Total time": total_time,
        "Average time": average_time,
        "Requests per second": requests_per_second
    }

stress_test_results = run_load_stress_test(number_of_requests=100)


Load and Stress Test Results
 Total time: 2.125 seconds
 Average time: 0.02125 seconds/request
 Requests per second: 47.06


In [ ]:
import gradio as gr
import pandas as pd
import os
from datetime import datetime

def unique_values(col):
    if col in data.columns:
        return sorted(data[col].dropna().astype(str).unique().tolist())
    return ["Unknown"]

gender_choices = [g for g in unique_values("Gender") if g != "Other"]
education_choices = unique_values("Education_Level")
internship_choices = unique_values("Internship_Experience")

def gradio_predict(age, gender, education, gpa, internship_experience, years_since_graduation):

    input_data = {
        "Education_Level": education,
        "Gender": gender,
        "Age": age,
        "Years_Since_Graduation": years_since_graduation,
        "GPA": gpa,
        "Internship_Experience": internship_experience
    }

    for col in features:
        if col not in input_data:
            if col in categorical_features:
                input_data[col] = str(data[col].mode()[0])
            else:
                input_data[col] = float(data[col].median())

    result = predict_unemployment_probability(input_data)

    log_row = input_data.copy()
    log_row.update(result)
    log_row["Timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    log_file = "prediction_logs.csv"
    log_df = pd.DataFrame([log_row])

    if os.path.exists(log_file):
        log_df.to_csv(log_file, mode="a", header=False, index=False)
    else:
        log_df.to_csv(log_file, index=False)

    return (
        result["Employment Probability"],
        result["Unemployment Probability"],
        result["Risk Level"],
        result["Model Used"],
        result["Explanation"]
    )

gr.close_all()

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# AI-Based Unemployment Risk Prediction System")
    gr.Markdown("Enter basic graduate information to predict employment and unemployment probability.")

    with gr.Row():

        with gr.Column(scale=1):

            age = gr.Slider(18, 65, value=25, step=1, label="Age")

            gender = gr.Dropdown(
                choices=gender_choices,
                value=gender_choices[0],
                label="Gender"
            )

            education = gr.Dropdown(
                choices=education_choices,
                value=education_choices[0],
                label="Education Level"
            )

            gpa = gr.Slider(0.0, 4.0, value=3.0, step=0.01, label="GPA")

            internship = gr.Dropdown(
                choices=internship_choices,
                value=internship_choices[0],
                label="Internship Experience"
            )

            years = gr.Slider(0, 30, value=2, step=1, label="Years Since Graduation")

            btn = gr.Button("Predict", variant="primary")

        with gr.Column(scale=1):

            out1 = gr.Textbox(label="Employment Probability")
            out2 = gr.Textbox(label="Unemployment Probability")
            out3 = gr.Textbox(label="Risk Level")
            out4 = gr.Textbox(label="Model Used")
            out5 = gr.Textbox(label="Explanation", lines=6)

    btn.click(
        fn=gradio_predict,
        inputs=[age, gender, education, gpa, internship, years],
        outputs=[out1, out2, out3, out4, out5]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://148bce30cb6dc594df.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
